# M1-B1 — Modélisation (corrigé)

Tâches 4 à 8. Le pipeline final et les fonctions de découpage sont dans `features.py` et `train.py` :
ce notebook les réutilise pour ne pas dupliquer le code.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

engine = create_engine(os.environ["DATABASE_URL"])

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit, cross_validate

import train as T                                                                       # custom training module for model training and evaluation 
from model.train_baseline import FEATURES as V1_FEATURES, build_pipeline as build_v1    # import baseline model features and pipeline builder



In [2]:
import importlib
import inspect
import train as T

importlib.reload(T)

print(T.__file__)
print(inspect.getsource(T.split_train_test))

/home/a895279/projects_CISIA/M1/Dev-IA-parcours-m1-b1/train.py
def split_train_test(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Met de côté les TEST_MONTHS derniers mois d'arrivée comme jeu de test."""
    return split_by_last_months(df, TEST_MONTHS)



print(V1_FEATURES)
['hotel', 'arrival_date_month', 'market_segment', 'deposit_type', 'customer_type', 'country', 'lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'adr', 'previous_cancellations', 'total_of_special_requests', 'required_car_parking_spaces']

In [3]:
df = pd.read_sql(text("""
    SELECT h.code AS hotel, r.*
    FROM reservations AS r JOIN hotels AS h ON h.hotel_id = r.hotel_id
    ORDER BY r.arrival_date, r.reservation_id
"""), engine, parse_dates=["arrival_date"])

#print(df.head(5))
"info \n"
print(df.info())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78703 entries, 0 to 78702
Data columns (total 35 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   hotel                           78703 non-null  object        
 1   reservation_id                  78703 non-null  int64         
 2   hotel_id                        78703 non-null  int64         
 3   is_canceled                     78703 non-null  int64         
 4   lead_time                       78703 non-null  int64         
 5   arrival_date_year               78703 non-null  int64         
 6   arrival_date_month              78703 non-null  object        
 7   arrival_date_week_number        78703 non-null  int64         
 8   arrival_date_day_of_month       78703 non-null  int64         
 9   stays_in_weekend_nights         78703 non-null  int64         
 10  stays_in_week_nights            78703 non-null  int64         
 11  ad

In [4]:
# Split the dataset into training and testing sets
train, test = T.split_train_test(df)
y_train, y_test = train["is_canceled"], test["is_canceled"]

In [5]:
# *check the distribution of the target variable in the training and testing sets
print("--train---\n")
print(train.info())
print("--test---\n")
print(test.info())
print("--y_train---\n")
print(y_train.head(5))
print(y_train.info())
print("--y_test---\n")
print(y_test.head(5))
print(y_test.info())
print("--y_train value counts---\n")
print("y_train value counts:")
print(y_train.value_counts())
print("--y_test value counts---\n")
print("y_test value counts:")
print(y_test.value_counts())
print("-----\n")

--train---

<class 'pandas.core.frame.DataFrame'>
Index: 58792 entries, 0 to 58791
Data columns (total 35 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   hotel                           58792 non-null  object        
 1   reservation_id                  58792 non-null  int64         
 2   hotel_id                        58792 non-null  int64         
 3   is_canceled                     58792 non-null  int64         
 4   lead_time                       58792 non-null  int64         
 5   arrival_date_year               58792 non-null  int64         
 6   arrival_date_month              58792 non-null  object        
 7   arrival_date_week_number        58792 non-null  int64         
 8   arrival_date_day_of_month       58792 non-null  int64         
 9   stays_in_weekend_nights         58792 non-null  int64         
 10  stays_in_week_nights            58792 non-null  int64         


## Tâche 4 — Un premier modèle

Toutes les colonnes, encodage sommaire, dates laissées de côté.

In [16]:
# Entraîner un modèle rapide sur presque toutes les colonnes pour vérifier s'il y a une anomalie.
""" 
    applique pd.get_dummies(..., dtype=int).fillna(0) pour créer X_naif, puis évalue un RandomForestClassifier(n_estimators=50) en validation croisée 3 folds (accuracy). Résultat observé : array([1., 1., 1.]) (100 % sur les 3 folds), ce qui est suspect.
"""
X_naif = pd.get_dummies(
    train.drop(columns=["is_canceled", "arrival_date", "reservation_status_date", "reservation_id"]),
    dtype=int,
).fillna(0)
rf = RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=0)         # Initialisation du classifieur Random Forest avec 50 arbres et parallélisation sur tous les cœurs disponibles.
#cross_validate(rf, X_naif, y_train, cv=3, scoring="accuracy")["test_score"]     # Évaluation du modèle Random Forest avec validation croisée à 3 plis et affichage des scores de test.

cv_results=cross_validate(rf, X_naif, y_train, cv=3, scoring="accuracy")["test_score"] 

sorted(cv_results)

[np.float64(1.0), np.float64(1.0), np.float64(1.0)]

In [ ]:
# Afficher la documentation de la fonction cross_validate pour comprendre ses paramètres et son fonctionnement.
help(cross_validate)
#print(cross_validate.help)

Multiple metric evaluation using ``cross_validate``
    (please refer the ``scoring`` parameter doc for more information)

    >>> scores = cross_validate(lasso, X, y, cv=3,
    ...                         scoring=('r2', 'neg_mean_squared_error'),
    ...                         return_train_score=True)
    >>> print(scores['test_neg_mean_squared_error'])
    [-3635.5... -3573.3... -6114.7...]
    >>> print(scores['train_r2'])
    [0.28009951 0.3908844  0.22784907]

In [7]:
# *Objectif : Identifier quelles colonnes expliquent ce score de 100 % :it the random forest model and display the top 5 feature importances
print("--fitting random forest model---\n")
"""
    rf : la forêt aléatoire définie dans la cellule précédente.
    X_naif : les caractéristiques numériques et encodées des réservations.
    y_train : leur résultat réel, 0 ou 1.
    .fit(...) : apprend à relier les caractéristiques au résultat.
"""

rf.fit(X_naif, y_train) # Entraîne le modèle Random Forest sur les données encodées et les résultats réels.

print("--top 5 feature importances---\n")
# a. Récupérer les importances des caractéristiques
print(pd.Series(data=rf.feature_importances_, index=X_naif.columns).sort_values(ascending=False).head(5))


--fitting random forest model---

--top 5 feature importances---

reservation_status_Check-Out    0.417759
reservation_status_Canceled     0.264817
deposit_type_No Deposit         0.059809
deposit_type_Non Refund         0.054535
country_PRT                     0.028015
dtype: float64


In [8]:
print(train.info())

<class 'pandas.core.frame.DataFrame'>
Index: 58792 entries, 0 to 58791
Data columns (total 35 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   hotel                           58792 non-null  object        
 1   reservation_id                  58792 non-null  int64         
 2   hotel_id                        58792 non-null  int64         
 3   is_canceled                     58792 non-null  int64         
 4   lead_time                       58792 non-null  int64         
 5   arrival_date_year               58792 non-null  int64         
 6   arrival_date_month              58792 non-null  object        
 7   arrival_date_week_number        58792 non-null  int64         
 8   arrival_date_day_of_month       58792 non-null  int64         
 9   stays_in_weekend_nights         58792 non-null  int64         
 10  stays_in_week_nights            58792 non-null  int64         
 11  adults 

**Diagnostic.** `reservation_status` est le *dernier* statut de la réservation (`Canceled`, `Check-Out`, `No-Show`) : c'est la réponse elle-même, connue **après** l'événement à prédire. Même raisonnement pour :

- `reservation_status_date` : la date de ce dernier statut ;
- `assigned_room_type` : la chambre attribuée à l'arrivée ;
- `booking_changes` : nombre de modifications, qui peut être mis à jour après la réservation : exclue par prudence ;
- `days_in_waiting_list` et `agent`/`company` sont connus à la réservation : pas de fuite.

Ces colonnes sont exclues de la suite. Dans `train.py`, elles ne sont même pas chargées.

### _conclusion ncmr_

Les deux indicateurs du statut final cumulent environ 68,26 % de l’importance totale attribuée par la forêt.

reservation_status_Check-Out = 1 indique un départ effectué.

reservation_status_Canceled = 1 indique une réservation annulée. 

Le modèle reçoit donc une information qui révèle déjà l’issue qu’il est censé prédire. C’est la fuite de données recherchée dans la tâche 4.

Les importances affichées ne permettent pas de dire, par exemple, « être portugais augmente le risque d’annulation » : elles ne donnent ni le sens de l’effet ni une relation causale.

Enfin, dtype: float64 signifie simplement que les importances sont stockées comme nombres décimaux.

Conclusion à noter : le modèle naïf s’appuie principalement sur des informations de statut final indisponibles au moment de la prédiction. Il faudra les exclure des modèles suivants.

## Tâche 5 — Reproduire la v1 (référence à battre)

On s'entraîne sur la recette de la v1:

Étape 1 — Validation croisée. Évalue build_pipeline() (importé de model/train_baseline.py) avec cross_validate, 5 folds, F1, précision et rappel, et return_train_score=True.

✅ Tu obtiens 5 scores par métrique. Note la moyenne et l'écart-type.

Étape 2 — Surapprentissage ? Compare train_f1 et test_f1 (dans cross_validate, « test » désigne le fold de validation, pas ton jeu de test). L'écart est-il important ?

Étape 3 — Une mini-recherche. Teste 2 valeurs de classifier__max_iter et 2 valeurs de classifier__learning_rate avec GridSearchCV. Affiche cv_results_ sous forme de DataFrame, en ne gardant que les paramètres, le score moyen et l'écart-type.

✅ Un tableau de 4 lignes, une par combinaison.

Étape 4 — TimeSeriesSplit. Refais l'étape 1 avec cv=TimeSeriesSplit(n_splits=5), après avoir trié tes données par date d'arrivée. Les scores changent-ils ? Pourquoi, à ton avis ?

In [ ]:
# Import des fonctions et constantes pour la version 1 du modèle (V1)
from model.train_baseline import (
    build_pipeline as build_v1,
    FEATURES as V1_FEATURES,
    TARGET as V1_TARGET,
)
from sklearn.model_selection import (
    GridSearchCV,
    TimeSeriesSplit,
    cross_validate,
)

In [11]:
# Préparation des données pour la version 1 du modèle (V1)

train_v1 = train.sort_values(
    ["arrival_date", "reservation_id"]
).copy()

X_train_v1 = train_v1[V1_FEATURES]          # X_train_v1 contient uniquement les 15 variables attendues par la v1.
y_train_v1 = train_v1[V1_TARGET]            # y_train_v1 contient les réponses réelles.

assert not X_train_v1.empty                 # X_train_v1 ne doit pas être vide.
assert X_train_v1.columns.is_unique         # Les colonnes de X_train_v1 doivent être uniques.
assert y_train_v1.notna().all()             # La cible ne doit pas contenir de valeurs manquantes.
assert set(y_train_v1.unique()) == {0, 1}   # La cible doit contenir uniquement 0 et 1.

print("------------\n")
print("Dimensions :", X_train_v1.shape)
print("------------\n")
print("Variables utilisées :", list(X_train_v1.columns))
print("------------\n")
print("Cible :", V1_TARGET)
print("------------\n")
#print("train_v1 :", train_v1.info())

------------

Dimensions : (58792, 15)
------------

Variables utilisées : ['hotel', 'arrival_date_month', 'market_segment', 'deposit_type', 'customer_type', 'country', 'lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'adr', 'previous_cancellations', 'total_of_special_requests', 'required_car_parking_spaces']
------------

Cible : is_canceled
------------

<class 'pandas.core.frame.DataFrame'>
Index: 58792 entries, 0 to 58791
Data columns (total 35 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   hotel                           58792 non-null  object        
 1   reservation_id                  58792 non-null  int64         
 2   hotel_id                        58792 non-null  int64         
 3   is_canceled                     58792 non-null  int64         
 4   lead_time                       58792 non-null  int64         
 5   arrival_date_year  

*X_train_v1.shape = dimensions du jeu de données d'entraînement pour la version 1 du modèle.*

Dimensions : (58792, 15)


*list(X_train_v1.columns) = colonnes utilisées pour la version 1 du modèle.*

**Variables utilisées** : ['hotel', 'arrival_date_month', 'market_segment', 'deposit_type', 'customer_type', 'country', 'lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'adr', 'previous_cancellations', 'total_of_special_requests', 'required_car_parking_spaces']

*V1_TARGET = variable cible pour la version 1 du modèle.*

**Cible** : is_canceled

In [ ]:
# 2. Évaluer la recette avec cinq folds
"""
    build_v1(),             # Construire le pipeline pour la version 1 du modèle
    X_train_v1,             # Données d'entraînement pour la version 1 du modèle
    y_train_v1,              # Cible pour la version 1 du modèle
    cv=5,                     # Nombre de folds pour la validation croisée  
    scoring=metriques,          # Métriques utilisées pour évaluer le modèle
    return_train_score=True,    # Inclure les scores d'entraînement pour diagnostiquer le surapprentissage
    n_jobs=1,                    # Nombre de jobs parallèles (1 pour éviter le multi-threading dans l'environnement partagé)
    error_score="raise",        # Remonter immédiatement toute erreur survenue lors de l’entraînement d’un fold
"""
metriques = ["f1", "precision", "recall", "roc_auc"]

# scoring : liste des métriques à utiliser pour évaluer le modèle
scores_v1 = cross_validate(
    build_v1(),
    X_train_v1,
    y_train_v1,
    cv=5,
    scoring=metriques,
    return_train_score=True,
    n_jobs=1,
    error_score="raise",
)

resultats_v1 = pd.DataFrame(scores_v1)

display(resultats_v1)


,fit_time,score_time,test_f1,train_f1,test_precision,train_precision,test_recall,train_recall,test_roc_auc,train_roc_auc
0,3.586094,0.630961,0.622230,0.874181,0.494366,0.900177,0.839311,0.849644,0.799647,0.972921
1,4.057251,0.282478,0.818087,0.860570,0.895845,0.883806,0.752750,0.838525,0.916254,0.968364
2,3.238146,0.242054,0.741689,0.878584,0.886303,0.909143,0.637646,0.850012,0.905303,0.976962
3,3.885458,0.258571,0.796493,0.874029,0.823544,0.905901,0.771162,0.844324,0.938264,0.973891
4,3.889376,0.273509,0.523933,0.893272,0.809191,0.923308,0.387374,0.865128,0.827491,0.980350


In [18]:
metriques = ["f1", "precision", "recall", "roc_auc"]

scores_v2 = cross_validate(
    build_v1(),
    X_train_v1,
    y_train_v1,
    cv=3,
    scoring=metriques,
    return_train_score=True,
    n_jobs=1,
    error_score="raise",
)

resultats_v2 = pd.DataFrame(scores_v2)

display(resultats_v2)

,fit_time,score_time,test_f1,train_f1,test_precision,train_precision,test_recall,train_recall,test_roc_auc,train_roc_auc
0,2.973105,0.471495,0.717187,0.866315,0.647842,0.890843,0.803156,0.843102,0.867119,0.969193
1,3.838210,0.803381,0.751039,0.885230,0.893020,0.911856,0.648013,0.860115,0.894211,0.978644
2,2.520046,0.426236,0.641441,0.897619,0.803325,0.929334,0.533859,0.867996,0.850990,0.982770


Que fait cet appel ?
Il entraîne cinq pipelines distincts :

chacun apprend sur quatre folds ;
chacun est évalué sur le cinquième ;
l’imputation et l’encodage sont réappris uniquement sur l’apprentissage de chaque fold.
return_train_score=True ajoute les scores d’entraînement, utiles pour diagnostiquer le surapprentissage.

Attention : pour cette classification, cv=5 utilise par défaut cinq folds stratifiés sans mélange. Ce n’est ni une validation aléatoire mélangée, ni une validation temporelle.

n_jobs=1 évite de lancer plusieurs entraînements simultanés dans l’environnement partagé.

error_score="raise" permet de remonter immédiatement toute erreur survenue lors de l’entraînement d’un fold, plutôt que de la masquer.

In [ ]:
print("mean")
print(pd.DataFrame(scores_v1).mean())

print("std")
print(pd.DataFrame(scores_v1).std())  


mean
fit_time           3.347218
score_time         0.324385
test_f1            0.700486
train_f1           0.876127
test_precision     0.781850
train_precision    0.904467
test_recall        0.677649
train_recall       0.849527
test_roc_auc       0.877392
train_roc_auc      0.974498
dtype: float64
std
fit_time           0.379205
score_time         0.097205
test_f1            0.124539
train_f1           0.011728
test_precision     0.165105
train_precision    0.014354
test_recall        0.177749
train_recall       0.009897
test_roc_auc       0.060268
train_roc_auc      0.004494
dtype: float64


In [ ]:
# Résumer les scores : la moyenne et l’écart-type des cinq scores pour chaque métrique.
colonnes_scores = [
    f"{jeu}_{metrique}"
    for jeu in ["train", "test"]
    for metrique in metriques
]

resume_v1 = resultats_v1[colonnes_scores].agg(["mean", "std"]).T

display(resume_v1.round(6))

,mean,std
train_f1,0.876127,0.011728
train_precision,0.904467,0.014354
train_recall,0.849527,0.009897
train_roc_auc,0.974498,0.004494
test_f1,0.700486,0.124539
test_precision,0.781850,0.165105
test_recall,0.677649,0.177749
test_roc_auc,0.877392,0.060268


| Metric          | Mean     | Std      ||
|-----------------|----------|----------|--|
| train_f1        | 0.876127 | 0.011728 ||
| train_precision | 0.904467 | 0.014354 ||
| train_recall    | 0.849527 | 0.009897 ||
| train_roc_auc   | 0.974498 | 0.004494 ||
| test_f1         | 0.700486 | 0.124539 |test_f1 à comparer avec train_f1|
| test_precision  | 0.781850 | 0.165105 ||
| test_recall     | 0.677649 | 0.177749 ||
| test_roc_auc    | 0.877392 | 0.060268 ||


In [ ]:
#  Comparer entraînement et validation pour la version 1 du modèle
diagnostic_v1 = resultats_v1[["train_f1", "test_f1"]].copy()

diagnostic_v1["ecart_f1"] = (
    diagnostic_v1["train_f1"] - diagnostic_v1["test_f1"]
)

display(diagnostic_v1.round(4))
display(diagnostic_v1.mean().to_frame("moyenne").round(6))

,train_f1,test_f1,ecart_f1
0,0.8742,0.6222,0.2520
1,0.8606,0.8181,0.0425
2,0.8786,0.7417,0.1369
3,0.8740,0.7965,0.0775
4,0.8933,0.5239,0.3693


,moyenne
train_f1,0.876127
test_f1,0.700486
ecart_f1,0.175641


Comment interpréter ?
Observation|	Interprétation possible|
--|--|
F1 d’entraînement nettement supérieur au F1 de validation|	Surapprentissage possible|
Scores proches et satisfaisants|	Meilleure généralisation apparente|
Scores proches mais faibles|	Sous-apprentissage possible ou variables insuffisantes|
Scores très variables selon les folds|	Instabilité, saisonnalité ou populations différentes|

Il n’existe pas de seuil universel définissant un « grand écart ». Examinez aussi la dispersion et les différences entre périodes.

In [ ]:
# 4. Tester une mini-grille de quatre configurations
"""
Objectif : Tester l'effet de classifier__max_iter ([100, 200]) et classifier__learning_rate ([0.05, 0.1]).
Fonctionnement : Exécute GridSearchCV(..., cv=5, scoring="f1", return_train_score=True, refit=False) soit 4 × 5 = 20 entraînements sans ré-entraîner le modèle final sur tout train (refit=False).
"""
grille_v1 = {
    "classifier__max_iter": [100, 200],
    "classifier__learning_rate": [0.05, 0.1],
}

recherche_v1 = GridSearchCV(
    estimator=build_v1(),
    param_grid=grille_v1,
    scoring="f1",
    cv=5,
    return_train_score=True,
    n_jobs=1,
    error_score="raise",
    refit=False,
)

recherche_v1.fit(X_train_v1, y_train_v1)


GridSearchCV(cv=5, error_score='raise',
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(sparse_threshold=0,
                                                          transformers=[('cat',
                                                                         Pipeline(steps=[('impute',
                                                                                          SimpleImputer(fill_value='Unknown',
                                                                                                        strategy='constant')),
                                                                                         ('encode',
                                                                                          OneHotEncoder(handle_unknown='infrequent_if_exist',
                                                                                                        min_frequency=50))]),
                                                                         ['hotel',
                                                                          'arrival_date_month',
                                                                          'market_segment'...
                                                                          'stays_in_week_nights',
                                                                          'adults',
                                                                          'children',
                                                                          'adr',
                                                                          'previous_cancellations',
                                                                          'total_of_special_requests',
                                                                          'required_car_parking_spaces'])])),
                                       ('classifier',
                                        HistGradientBoostingClassifier(max_iter=200,
                                                                       random_state=42))]),
             n_jobs=1,
             param_grid={'classifier__learning_rate': [0.05, 0.1],
                         'classifier__max_iter': [100, 200]},
             refit=False, return_train_score=True, scoring='f1')

classifier__ désigne l’étape nommée classifier dans le pipeline.

Deux valeurs pour chaque paramètre donnent :

[
2 \times 2 = 4\ configurations,\quad 4 \times 5 = 20\ entraînements.
]

refit=False évite ici un entraînement supplémentaire sur tout train : nous voulons d’abord examiner les essais.

In [ ]:
# Afficher les quatre lignes demandées (trace GridSearchCV)
""" 
Objectif : Afficher les 4 configurations triées par mean_test_score.
Fonctionnement : Extrait recherche_v1.cv_results_ dans trace_v1, vérifie assert len(trace_v1) == 4 (meilleur F1 moyen : 0.7077 avec max_iter=100, learning_rate=0.05), et rappelle de ne pas remplacer silencieusement la référence v1 initiale par cette variante.
"""

trace_v1 = pd.DataFrame(recherche_v1.cv_results_)[[
    "param_classifier__max_iter",
    "param_classifier__learning_rate",
    "mean_train_score",
    "mean_test_score",
    "std_test_score",
]].sort_values("mean_test_score", ascending=False)

assert len(trace_v1) == 4

display(trace_v1)
print("Meilleurs paramètres explorés :", recherche_v1.best_params_)
print("Meilleur F1 moyen :", recherche_v1.best_score_)

,param_classifier__max_iter,param_classifier__learning_rate,mean_train_score,mean_test_score,std_test_score
0,100,0.05,0.847291,0.707716,0.102686
2,100,0.10,0.859047,0.707149,0.108487
1,200,0.05,0.857905,0.706495,0.107663
3,200,0.10,0.876127,0.700486,0.111391


Meilleurs paramètres explorés : {'classifier__learning_rate': 0.05, 'classifier__max_iter': 100}
Meilleur F1 moyen : 0.7077162726351545


- les premiers résultats évaluent la recette v1 d’origine ;
- cette grille constitue une expérience de réglage, à identifier séparément.
- 
Ne remplacez pas silencieusement votre référence initiale par sa variante optimisée.

 **le compromis observé entre score, dispersion et écart entraînement/validation? :**


- Un score moyen élevé avec une faible dispersion et un écart raisonnable entre entraînement et validation indique une bonne généralisation.
- Un score moyen élevé mais avec une forte dispersion ou un écart important peut signaler un surapprentissage ou une instabilité.
- Un score moyen faible avec peu de dispersion peut indiquer un sous-apprentissage ou des variables insuffisantes.
- Un score moyen faible avec une forte dispersion peut signaler une instabilité ou des données bruitées.

In [ ]:
# 5. Refaire l’évaluation avec TimeSeriesSplit : Pour isoler l’effet du découpage, reprenez la recette initiale, pas les meilleurs paramètres de la grille.
""" 
Validation croisée temporelle (TimeSeriesSplit)
Objectif : Évaluer build_v1() en respectant l'ordre chronologique.
Fonctionnement : Utilise cv_temporel = TimeSeriesSplit(n_splits=5) sur X_train_v1 trié par date et stocke les résultats dans resultats_v1_temporels. La cellule Markdown rappelle les deux limites : premiers folds avec moins de données, et coupure par nombre de lignes (un même jour peut être à cheval).
"""
cv_temporel = TimeSeriesSplit(n_splits=5)

scores_v1_temporels = cross_validate(
    build_v1(),
    X_train_v1,
    y_train_v1,
    cv=cv_temporel,
    scoring=metriques,
    return_train_score=True,
    n_jobs=1,
    error_score="raise",
)

resultats_v1_temporels = pd.DataFrame(scores_v1_temporels)

display(resultats_v1_temporels)

,fit_time,score_time,test_f1,train_f1,test_precision,train_precision,test_recall,train_recall,test_roc_auc,train_roc_auc
0,0.651240,0.131144,0.871532,0.960676,0.899073,0.957672,0.845627,0.963698,0.966401,0.996127
1,0.942307,0.141768,0.717283,0.949043,0.898848,0.960150,0.596742,0.938189,0.854997,0.994640
2,2.790752,0.394263,0.704178,0.919551,0.845662,0.947536,0.603252,0.893171,0.894919,0.988676
3,5.432626,0.240586,0.788265,0.898276,0.863356,0.930211,0.725191,0.868461,0.926821,0.982881
4,3.211180,0.232906,0.423765,0.890964,0.757230,0.924322,0.294205,0.859930,0.810555,0.979753


Contrairement au premier protocole, on n’apprend pas sur des lignes postérieures à celles de validation.

Deux limites à connaître :

les premiers folds disposent de moins de données d’apprentissage ;
le découpage se fait par nombre de lignes : un même jour peut être partagé entre apprentissage et validation. Pour des dates strictement disjointes, il faudrait définir des frontières calendaires explicites.

En pratique, TimeSeriesSplit est utile pour simuler un scénario de prévision où l’on ne peut utiliser que les informations passées pour prédire le futur.
Il permet ainsi d’évaluer la robustesse d’un modèle dans des conditions réalistes de prévision temporelle.
Cela est particulièrement pertinent pour les séries temporelles financières, météorologiques ou toute autre donnée où l’ordre chronologique est crucial.
En résumé, TimeSeriesSplit offre une évaluation plus réaliste pour les modèles destinés à la prévision temporelle, en respectant la contrainte fondamentale de ne pas utiliser le futur pour prédire le passé.

In [ ]:
# 6. Comparaison stratifié vs temporel
""" 
Objectif : Comparer moyennes et écarts-types des deux protocoles de validation.
Fonctionnement : Construit comparaison pour les 4 métriques test_* (en temporel : F1 moyen quasi identique à 0.7010 mais écart-type plus élevé à 0.1687, précision plus haute à 0.8528, rappel plus bas à 0.6130).
"""

comparaison = pd.DataFrame({
    "stratifie_moyenne": resultats_v1.mean(),
    "stratifie_ecart_type": resultats_v1.std(),
    "temporel_moyenne": resultats_v1_temporels.mean(),
    "temporel_ecart_type": resultats_v1_temporels.std(),
})

display(
    comparaison.loc[
        [f"test_{metrique}" for metrique in metriques]
    ].round(6)
)


,stratifie_moyenne,stratifie_ecart_type,temporel_moyenne,temporel_ecart_type
test_f1,0.700486,0.124539,0.701005,0.168694
test_precision,0.781850,0.165105,0.852834,0.058219
test_recall,0.677649,0.177749,0.613003,0.205361
test_roc_auc,0.877392,0.060268,0.890739,0.060727


Les différences peuvent venir :

- d’une évolution des comportements dans le temps ;
- de la saisonnalité ;
- de catégories nouvelles dans les périodes suivantes ;
- de la quantité d’apprentissage différente entre les protocoles.
- de la présence de bruit ou d’événements exceptionnels dans certaines périodes.
- 
Une baisse des scores temporels est possible, pas obligatoire. Puisque votre objectif est de prédire des réservations futures, le protocole temporel est le plus pertinent pour orienter la suite.

## Tâche 6 — Pipeline et variables

Les variables calculées sont dans `features.py` (fonction nommée, importable par l'API).
Comparaison de trois modèles, avec les mêmes variables.

In [19]:
# Préparation des données pour la version 1 du modèle (V1)

# Donnees brutes
train_v6 = train.sort_values(
    ["arrival_date", "reservation_id"]
).copy()

X_train_v6 = train_v6[V1_FEATURES]          
y_train_v6 = train_v6[V1_TARGET]            

# prep 
assert not X_train_v6.empty                 
assert X_train_v6.columns.is_unique         
assert y_train_v6.notna().all()             
assert set(y_train_v6.unique()) == {0, 1}  

In [ ]:
# prep étape 2

X_train_v6 = X_train_v6.drop(
    columns=["country", "children"]
)

stays_in_weekend_nights
stays_in_week_nights

## Tâche 7 — Valider et régler

D'abord avec une validation croisée **aléatoire**, puis **temporelle**

**Choix du modèle.** 

## Tâche 8 — Le verdict

**Coût des erreurs pour l'hôtel :**

- annulation **ratée** (faux négatif) : la chambre reste vide, nuitée perdue. C'est la plainte du service réservations ;
- annulation **prédite à tort** (faux positif) : si l'hôtel survend la chambre, le client arrive et doit être relogé. Coût élevé, mais seulement si l'hôtel agit sur toutes les alertes ; s'il se contente de relancer le client, une fausse alerte coûte peu.

→ **F1** comme métrique principale (équilibre), en suivant le **rappel** de près, puisque c'est le défaut constaté de la v1.

Le seuil de décision est choisi sur les 4 derniers mois de l'**entraînement** (jamais sur le test).

In [ ]:
seuil, trace_seuil = T.choose_threshold(train, best_C)


In [ ]:
v1 = build_v1().
v2_05 = T.build_pipeline(best_C, 0.5).
v2 = T.build_pipeline(best_C, seuil).



In [ ]:
ConfusionMatrixDisplay.from_estimator(v1, test[INPUT_FEATURES], y_test, display_labels=["maintenue", "annulée"]);
ConfusionMatrixDisplay.from_estimator(v2, test[INPUT_FEATURES], y_test, display_labels=["maintenue", "annulée"]);